# Grouped sensitivity plotting with automatic AS-generated share check

In [ ]:
"""
Grouped plots for sensitivity outputs, with configurable font sizes
and automatic AS-generated (ASG) share detection.

This script uses only the two summary files:
  - Sensitivity_Case_Level_Summary.csv
  - Sensitivity_Change_Summary_By_Case.csv

The plotting structure is grouped into three columns:
  1. Risk-preference sensitivity: SellerRisk, BuyerRisk, BothRisk
  2. Penalty-scale sensitivity: Gamma
  3. Residual-scale robustness: VolumeVol, PriceVol

The baseline is inserted into each group:
  - x = 0 for risk-preference cases, because baseline has lambda_s=lambda_b=0
  - x = 1 for gamma/residual-scale cases, because baseline scale equals 1

Main outputs:
  - Fig_sensitivity_stability_participation_grouped.png
  - Fig_sensitivity_contract_terms_pct_change_grouped.png
  - Fig_sensitivity_contract_mix_grouped.png

Contract-form plotting logic:
  - Fixed-volume and ASC shares are always checked and plotted.
  - ASG share is checked automatically.
  - If ASG share is positive in any case, a third row of subfigures is added.
  - If ASG share is zero in all cases, the figure remains two rows.

Label convention:
  - share_same_full_decision is plotted as "Baseline-identical outcome rate"
  - ppa_share is plotted as "PPA formation rate"

Display convention:
  - share-metric axes extend slightly below 0%, while tick labels remain 0--100%
  - font sizes are controlled by the FONT_SIZES dictionary below
"""

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

read_csv_optimized = pd.read_csv


# -----------------------------------------------------------------------------
# 1. User configuration
# -----------------------------------------------------------------------------
ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path(".")
FIG_DIR = ROOT / "Sensitivity_Figures_Grouped_v4"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CASE_SUMMARY_FILE = "Output files (Sensitivity, No Mutation, Verified)/Sensitivity_Case_Level_Summary.csv"
CHANGE_SUMMARY_FILE = "Output files (Sensitivity, No Mutation, Verified)/Sensitivity_Change_Summary_By_Case.csv"

SAVE_FIGURES = True
SHOW_FIGURES = True
DPI = 300

FONT_SIZES = {
    "tick": 14,
    "axis_label": 14,
    "subplot_title": 14,
    "legend": 14,
    "figure_title": 14,
}

SHOW_GROUP_TITLES = True
SHOW_FIGURE_TITLE = False
LEGEND_LOCATION = "best"

# Numerical tolerance for deciding whether a share is effectively zero.
ZERO_TOL = 1e-12


# -----------------------------------------------------------------------------
# 2. Load and merge summaries
# -----------------------------------------------------------------------------
def load_sensitivity_summary(root: Path) -> pd.DataFrame:
    case = read_csv_optimized(root / CASE_SUMMARY_FILE)
    change = read_csv_optimized(root / CHANGE_SUMMARY_FILE)

    change_cols = [
        "case_id",
        "share_same_full_decision",
        "share_same_ppa_type",
        "share_same_volume",
        "share_same_price",
        "mean_seller_utility_diff_vs_baseline",
        "mean_buyer_utility_diff_vs_baseline",
    ]
    change_cols = [c for c in change_cols if c in change.columns]

    df = case.merge(change[change_cols], on="case_id", how="left")
    df["case_value_numeric"] = pd.to_numeric(df["case_value_numeric"], errors="coerce")
    df["case_value"] = pd.to_numeric(df["case_value"], errors="coerce")

    base = df.loc[df["case_family"] == "Baseline"].iloc[0]
    df["strike_pct_change_vs_baseline"] = df["strike_mean"] / base["strike_mean"] - 1.0
    df["volume_pct_change_vs_baseline"] = df["volume_mean"] / base["volume_mean"] - 1.0
    return df


df = load_sensitivity_summary(ROOT)


# -----------------------------------------------------------------------------
# 3. Labels and grouped plotting data
# -----------------------------------------------------------------------------
FAMILY_LABELS = {
    "SellerRisk": r"Seller risk aversion $\lambda^S$",
    "BuyerRisk": r"Buyer risk aversion $\lambda^B$",
    "BothRisk": r"Joint risk aversion $\lambda^S=\lambda^B$",
    "Gamma": r"Penalty scale $\gamma$",
    "VolumeVol": r"Volume residual scale",
    "PriceVol": r"Price residual scale",
    "Baseline": "Baseline",
}

PLOT_GROUPS = {
    "risk": {
        "title": "Risk aversion sensitivity",
        "families": ["SellerRisk", "BuyerRisk", "BothRisk"],
        "baseline_x": 0.0,
        "xlabel": r"Risk aversion weight",
    },
    "penalty": {
        "title": "Penalty-scale sensitivity",
        "families": ["Gamma"],
        "baseline_x": 1.0,
        "xlabel": r"Penalty multiplier $\gamma$",
    },
    "residual": {
        "title": "Residual-scale robustness",
        "families": ["VolumeVol", "PriceVol"],
        "baseline_x": 1.0,
        "xlabel": r"Residual multiplier $\alpha$",
    },
}

# Displayed sensitivity columns; penalty-scale cases remain available in the data export.
FIGURE_GROUP_KEYS = ["risk", "residual"]


def grouped_data(source: pd.DataFrame, group_key: str) -> pd.DataFrame:
    spec = PLOT_GROUPS[group_key]
    base_rows = source.loc[source["case_family"] == "Baseline"]
    if base_rows.empty:
        raise ValueError("Baseline row not found in sensitivity summary.")
    base = base_rows.iloc[0].copy()

    pieces = []
    for order, fam in enumerate(spec["families"]):
        b = base.copy()
        b["case_family"] = fam
        b["case_label"] = "Baseline"
        b["case_id"] = f"BASE_FOR_{fam}"
        b["plot_x"] = spec["baseline_x"]
        b["family_order"] = order
        b["is_baseline_reference"] = True
        pieces.append(pd.DataFrame([b]))

        sub = source.loc[source["case_family"] == fam].copy()
        sub["plot_x"] = pd.to_numeric(sub["case_value_numeric"], errors="coerce")
        sub["family_order"] = order
        sub["is_baseline_reference"] = False
        pieces.append(sub)

    out = pd.concat(pieces, ignore_index=True)
    out["family_label"] = out["case_family"].map(FAMILY_LABELS).fillna(out["case_family"])
    out = out.sort_values(["family_order", "plot_x", "case_order", "case_id"]).reset_index(drop=True)
    return out


def apply_axis_font_sizes(ax):
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZES["tick"])
    ax.tick_params(axis="both", which="minor", labelsize=FONT_SIZES["tick"])


def add_y_format(ax, metric_type: str):
    if metric_type in {"share", "pct_change"}:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    if metric_type == "share":
        ax.set_ylim(-0.035, 1.05)
        ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    if metric_type == "pct_change":
        ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.8)


def plot_lines(ax, data: pd.DataFrame, y_col: str, metric_type: str = "value"):
    markers = ["o", "s", "^", "D", "v", "P"]
    ordered = data.sort_values(["family_order", "plot_x"])
    for i, (label, sub) in enumerate(ordered.groupby("family_label", sort=False)):
        sub = sub.sort_values("plot_x")
        ax.plot(
            sub["plot_x"],
            sub[y_col],
            marker=markers[i % len(markers)],
            linewidth=2.0,
            markersize=5.5,
            label=label,
        )
    ax.grid(True, alpha=0.25)
    add_y_format(ax, metric_type)
    apply_axis_font_sizes(ax)


def set_axis_text(ax, xlabel: str = "", ylabel: str = "", title: str = ""):
    ax.set_xlabel(xlabel, fontsize=FONT_SIZES["axis_label"])
    ax.set_ylabel(ylabel, fontsize=FONT_SIZES["axis_label"])
    ax.set_title(title, fontsize=FONT_SIZES["subplot_title"])


def add_legend(ax):
    return ax.legend(loc=LEGEND_LOCATION, fontsize=FONT_SIZES["legend"], frameon=True)


def savefig(fig, filename: str):
    if SAVE_FIGURES:
        path = FIG_DIR / filename
        fig.savefig(path, dpi=DPI, bbox_inches="tight")
        print(f"Saved: {path}")


# -----------------------------------------------------------------------------
# 4. ASG presence check
# -----------------------------------------------------------------------------
def asg_share_present(source: pd.DataFrame) -> bool:
    if "asg_share" not in source.columns:
        return False
    vals = pd.to_numeric(source["asg_share"], errors="coerce").fillna(0.0)
    return bool((vals.abs() > ZERO_TOL).any())


def export_asg_check(source: pd.DataFrame):
    has_asg = asg_share_present(source)
    vals = pd.to_numeric(source.get("asg_share", 0.0), errors="coerce").fillna(0.0)

    summary = pd.DataFrame([
        {
            "asg_share_present": has_asg,
            "max_asg_share": float(vals.max()) if len(vals) else 0.0,
            "n_cases_with_positive_asg_share": int((vals > ZERO_TOL).sum()) if len(vals) else 0,
        }
    ])
    summary.to_csv(FIG_DIR / "ASG_presence_check.csv", index=False)

    if has_asg:
        details = source.loc[vals > ZERO_TOL, [c for c in ["case_id", "case_label", "case_family", "case_value_numeric", "asg_share"] if c in source.columns]].copy()
        details.to_csv(FIG_DIR / "ASG_positive_cases.csv", index=False)
        print("AS-generated contracts detected. Added ASG row to contract-mix figure.")
        print(f"Saved: {FIG_DIR / 'ASG_positive_cases.csv'}")
    else:
        print("No AS-generated contracts detected in the current sensitivity outputs. Contract-mix figure remains two rows.")

    print(f"Saved: {FIG_DIR / 'ASG_presence_check.csv'}")
    return has_asg


HAS_ASG = export_asg_check(df)


# -----------------------------------------------------------------------------
# 5. Figure A: outcome stability and participation
# -----------------------------------------------------------------------------
def plot_stability_and_participation():
    metrics = [
        ("share_same_full_decision", "Reference-identical outcome rate", "share"),
        ("ppa_share", "PPA-selection share", "share"),
    ]
    group_keys = FIGURE_GROUP_KEYS

    fig, axes = plt.subplots(
        nrows=len(metrics),
        ncols=len(group_keys),
        figsize=(5.0 * len(group_keys), 7),
        sharey="row",
        constrained_layout=True,
    )

    for col, group_key in enumerate(group_keys):
        gdf = grouped_data(df, group_key)
        spec = PLOT_GROUPS[group_key]
        for row, (y_col, y_label, metric_type) in enumerate(metrics):
            ax = axes[row, col]
            plot_lines(ax, gdf, y_col, metric_type=metric_type)

            group_title = spec["title"] if (row == 0 and SHOW_GROUP_TITLES) else ""
            xlabel = spec["xlabel"] if row == len(metrics) - 1 else ""
            ylabel = y_label if col == 0 else ""
            set_axis_text(ax, xlabel=xlabel, ylabel=ylabel, title=group_title)

            if row == 0:
                add_legend(ax)

    if SHOW_FIGURE_TITLE:
        fig.suptitle(
            "Sensitivity diagnostics: outcome stability and PPA formation",
            fontsize=FONT_SIZES["figure_title"],
        )

    savefig(fig, "Fig_sensitivity_stability_participation_grouped.png")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# -----------------------------------------------------------------------------
# 6. Figure B: contract terms as percentage changes from baseline
# -----------------------------------------------------------------------------
def plot_contract_terms_pct_change():
    metrics = [
        ("strike_pct_change_vs_baseline", "Mean strike price change", "pct_change"),
        ("volume_pct_change_vs_baseline", "Mean contracted volume change", "pct_change"),
    ]
    group_keys = FIGURE_GROUP_KEYS

    fig, axes = plt.subplots(
        nrows=len(metrics),
        ncols=len(group_keys),
        figsize=(5.0 * len(group_keys), 7),
        sharey="row",
        constrained_layout=True,
    )

    for col, group_key in enumerate(group_keys):
        gdf = grouped_data(df, group_key)
        spec = PLOT_GROUPS[group_key]
        for row, (y_col, y_label, metric_type) in enumerate(metrics):
            ax = axes[row, col]
            plot_lines(ax, gdf, y_col, metric_type=metric_type)

            group_title = spec["title"] if (row == 0 and SHOW_GROUP_TITLES) else ""
            xlabel = spec["xlabel"] if row == len(metrics) - 1 else ""
            ylabel = y_label if col == 0 else ""
            set_axis_text(ax, xlabel=xlabel, ylabel=ylabel, title=group_title)

            if row == 0:
                add_legend(ax)

    if SHOW_FIGURE_TITLE:
        fig.suptitle(
            "Sensitivity diagnostics: contract-term changes relative to baseline",
            fontsize=FONT_SIZES["figure_title"],
        )

    savefig(fig, "Fig_sensitivity_contract_terms_pct_change_grouped.png")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# -----------------------------------------------------------------------------
# 7. Figure C: contract-form shares, with optional ASG row
# -----------------------------------------------------------------------------
def plot_contract_mix():
    metrics = [
        ("fix_share", "Fixed-Volume contract rate", "share"),
        ("asc_share", "As-Consumed contract rate", "share"),
    ]
    if HAS_ASG:
        metrics.append(("asg_share", "AS-generated contract rate", "share"))

    group_keys = FIGURE_GROUP_KEYS
    fig_height = 7 if len(metrics) == 2 else 10

    fig, axes = plt.subplots(
        nrows=len(metrics),
        ncols=len(group_keys),
        figsize=(5.0 * len(group_keys), fig_height),
        sharey="row",
        constrained_layout=True,
    )

    if len(metrics) == 1:
        axes = [axes]

    for col, group_key in enumerate(group_keys):
        gdf = grouped_data(df, group_key)
        spec = PLOT_GROUPS[group_key]
        for row, (y_col, y_label, metric_type) in enumerate(metrics):
            ax = axes[row, col]
            plot_lines(ax, gdf, y_col, metric_type=metric_type)

            group_title = spec["title"] if (row == 0 and SHOW_GROUP_TITLES) else ""
            xlabel = spec["xlabel"] if row == len(metrics) - 1 else ""
            ylabel = y_label if col == 0 else ""
            set_axis_text(ax, xlabel=xlabel, ylabel=ylabel, title=group_title)

            if row == 0:
                add_legend(ax)

    if SHOW_FIGURE_TITLE:
        fig.suptitle(
            "Sensitivity diagnostics: contract-form composition",
            fontsize=FONT_SIZES["figure_title"],
        )

    savefig(fig, "Fig_sensitivity_contract_mix_grouped.png")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# -----------------------------------------------------------------------------
# 8. Optional table for checking plotted values
# -----------------------------------------------------------------------------
def export_plotting_table():
    pieces = []
    for key in ["risk", "penalty", "residual"]:
        gdf = grouped_data(df, key).copy()
        gdf.insert(0, "plot_group", key)
        pieces.append(gdf)
    out = pd.concat(pieces, ignore_index=True)
    keep = [
        "plot_group",
        "family_label",
        "case_id",
        "case_label",
        "plot_x",
        "share_same_full_decision",
        "ppa_share",
        "fix_share",
        "asc_share",
        "asg_share",
        "strike_mean",
        "volume_mean",
        "strike_pct_change_vs_baseline",
        "volume_pct_change_vs_baseline",
    ]
    keep = [c for c in keep if c in out.columns]
    out[keep].to_csv(FIG_DIR / "Sensitivity_grouped_plot_values.csv", index=False)
    print(f"Saved: {FIG_DIR / 'Sensitivity_grouped_plot_values.csv'}")


# -----------------------------------------------------------------------------
# 9. Run all plots
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    plot_stability_and_participation()
    plot_contract_terms_pct_change()
    plot_contract_mix()
    export_plotting_table()


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))
